# Notebook 3

L'ultimo test è una verifica del comportamento di una rete preallenata (ResNet50) con fine tuning sul dataset.

In [ ]:
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import AveragePooling2D
from keras.layers import Dropout
from keras.applications import ResNet50
import pathlib
import keras

import tensorflow as tf

## Download dataset

In [ ]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,
        cache_dir = CACHE_DIR,
        untar     = True,
        extract   = True,
    )
)

### Parameters

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32
SEED       = 67
TRAIN_SPLIT = 0.75
VAL_SPLIT  = 0.15

VAL_SPLIT  = 0.2
AUTOTUNE = tf.data.AUTOTUNE

### Util functions

In [ ]:
full_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

total_batches = len(full_ds)

train_size = int(TRAIN_SPLIT * total_batches)
val_size   = int(VAL_SPLIT * total_batches)

train_ds = full_ds.take(train_size)
val_ds   = full_ds.skip(train_size).take(val_size)
test_ds  = full_ds.skip(train_size + val_size)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
baseModel = ResNet50(weights="imagenet", include_top=False,
	input_tensor=keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)))

headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(256, activation="relu")(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(5, activation="softmax")(headModel)

model = keras.models.Model(inputs=baseModel.input, outputs=headModel)

for layer in baseModel.layers:
	layer.trainable = False

In [ ]:
opt = keras.optimizers.Adam(learning_rate=1e-4, weight_decay=1e-4 / 20)
model.compile(loss="sparse_categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

H = model.fit(
	train_ds,
	validation_data=val_ds,
	epochs=20)

loss, accuracy = model.evaluate(test_ds)
print(f"Ensemble Accuracy: {accuracy:.4f}")

model.save('models/resnet50_fine_tuned.keras')